# Dashboard (Phase 5: anymap-ts)

This notebook subscribes to agent topics and visualizes live person state on a map.

Phase 5 scope:
- Subscribe to person state, camera decisions, and entry events
- Render live markers with white/green/red colors
- Track occupancy from entered/exited events
- Keep dashboard read-only (no control decisions)

## Run checklist
- Run Cells 3-4 to load config and render the map.
- Run Cell 5 and click **Run agents (camera → control → people)** to start publishing data.
- Run Cells 6-7 to connect dashboard MQTT and activate subscriptions.
- Optionally run Cell 8 for heartbeat logs during demos.

In [1]:
# Cell purpose: import libraries and load shared project configuration
from __future__ import annotations

import json
import time
import sys
import os
from pathlib import Path
from collections import defaultdict

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / "src"
    if (src_dir / "simulated_city").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

os.environ["SIMCITY_MQTT_PROFILES"] = "local"

from anymap_ts import Map
from simulated_city.config import load_config
import simulated_city.mqtt as mqtt

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

CENTER_LAT = sim_cfg.center_lat if sim_cfg else 55.6479
CENTER_LON = sim_cfg.center_lon if sim_cfg else 12.0417

if sim_cfg and sim_cfg.locations:
    gate_locations = [
        {"entry_id": location.location_id, "lat": location.lat, "lon": location.lon}
        for location in sim_cfg.locations
    ]
else:
    gate_locations = []

print(f"Loaded config. MQTT base topic: {mqtt_cfg.base_topic}")
print(f"Dashboard center: lat={CENTER_LAT}, lon={CENTER_LON}")
print(f"Gate markers configured: {len(gate_locations)}")

Loaded config. MQTT base topic: simulated-city/stadium
Dashboard center: lat=55.65, lon=12.4186
Gate markers configured: 4


In [2]:
# Cell purpose: create the anymap-ts map widget used for live visualization
map_view = Map(center=(CENTER_LON, CENTER_LAT), zoom=16.5, height="650px", width="100%")
map_view.add_basemap("OpenStreetMap.Mapnik")
print("Using basemap: OpenStreetMap.Mapnik")

COLOR_HEX = {
    "white": "#FFFFFF",
    "green": "#2E7D32",
    "red": "#C62828",
}
GATE_COLOR_HEX = "#1565C0"

gate_marker_ids: dict[str, str] = {}
for gate in gate_locations:
    gate_id = str(gate["entry_id"])
    gate_marker_ids[gate_id] = map_view.add_marker(
        float(gate["lon"]),
        float(gate["lat"]),
        name=f"gate-{gate_id}",
        color=GATE_COLOR_HEX,
        popup=f"Gate {gate_id}",
    )

map_file = Path.cwd() / "dashboard_map.html"
try:
    map_view.to_html(str(map_file), title="Simulated City Dashboard")
    print(f"Saved anymap HTML file to: {map_file}")
except PermissionError as exc:
    print(f"Map HTML export skipped ({exc})")

print(f"Map created with {len(gate_marker_ids)} gate markers (blue).")
map_view

Using basemap: OpenStreetMap.Mapnik
Saved anymap HTML file to: /Users/streuli11/Documents/GitHub/BIF_Project_Overv-gning/notebooks/dashboard_map.html
Map created with 4 gate markers (blue).


In [3]:
map_view.add_coordinates_control()


In [ ]:
# Cell purpose: provide a one-click runner for agent notebooks in startup order
from pathlib import Path
from threading import Thread
import time

import ipywidgets as widgets
import nbformat
from nbclient import NotebookClient

def _find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src" / "simulated_city").exists():
            return candidate
    return start.resolve()

def _run_code_cells(notebook_path: Path, max_code_cells: int, timeout_s: int = 180) -> None:
    nb = nbformat.read(notebook_path, as_version=4)
    client = NotebookClient(
        nb,
        timeout=timeout_s,
        resources={"metadata": {"path": str(notebook_path.parent)}},
    )
    with client.setup_kernel():
        code_seen = 0
        for index, cell in enumerate(nb.cells):
            if cell.get("cell_type") != "code":
                continue
            code_seen += 1
            if code_seen > max_code_cells:
                break
            client.execute_cell(cell, index)

runner_output = widgets.Output(layout={"border": "1px solid #ddd", "padding": "8px", "height": "220px", "overflow_y": "auto"})
run_button = widgets.Button(
    description="Run agents (camera → control → people)",
    button_style="primary",
    icon="play",
    tooltip="Run agent notebooks in startup order",
)

def _run_agents_in_order(_: widgets.Button) -> None:
    run_button.disabled = True

    def _worker() -> None:
        repo_root = _find_repo_root(Path.cwd())
        plan = [
            ("agent_camera.ipynb", 4),
            ("agent_control.ipynb", 4),
            ("agent_people.ipynb", 7),
        ]

        with runner_output:
            runner_output.clear_output()
            print("Starting agent notebooks...")
            print("Order: camera -> control -> people")
            print("")

        try:
            for notebook_name, code_cells in plan:
                notebook_path = repo_root / "notebooks" / notebook_name
                start = time.time()
                with runner_output:
                    print(f"Running {notebook_name} (first {code_cells} code cells)...")

                _run_code_cells(notebook_path, max_code_cells=code_cells)

                elapsed = time.time() - start
                with runner_output:
                    print(f"Done {notebook_name} in {elapsed:.1f}s")
                    print("")

            with runner_output:
                print("All agents started. Dashboard should now receive updates.")
        except Exception as exc:
            with runner_output:
                print(f"Run failed: {exc}")
        finally:
            run_button.disabled = False

    Thread(target=_worker, daemon=True).start()

run_button.on_click(_run_agents_in_order)
widgets.VBox([
    widgets.HTML("<b>Quick agent runner</b><br>Use this to start camera, control, and people in order."),
    run_button,
    runner_output,
    widgets.HTML("<small>Tip: After running this, execute Cells 6-7 in this dashboard notebook.</small>"),
])

In [4]:
# Cell purpose: initialize MQTT client, topics, and dashboard state stores
dashboard_client_suffix = f"dashboard-phase5-{int(time.time())}"
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix=dashboard_client_suffix)

person_state_topic = f"{mqtt_cfg.base_topic}/person/state"
camera_decision_topic = f"{mqtt_cfg.base_topic}/camera/decision"
entry_event_topic = f"{mqtt_cfg.base_topic}/entry/event"

latest_people: dict[str, dict] = {}
marker_ids: dict[str, str] = {}
decision_counts = defaultdict(int)
inside_count = 0
message_counts = defaultdict(int)
last_render_ts = time.time()

print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Dashboard client suffix: {dashboard_client_suffix}")
print(f"Subscribing to: {person_state_topic}")
print(f"Subscribing to: {camera_decision_topic}")
print(f"Subscribing to: {entry_event_topic}")

Connected to MQTT broker at 127.0.0.1:1883
Dashboard client suffix: dashboard-phase5-1772629283
Subscribing to: simulated-city/stadium/person/state
Subscribing to: simulated-city/stadium/camera/decision
Subscribing to: simulated-city/stadium/entry/event


In [5]:
# Cell purpose: define subscriber callbacks to update markers, decisions, and occupancy
def _safe_color(value: str) -> str:
    lowered = (value or "white").lower()
    return lowered if lowered in COLOR_HEX else "white"

def _render_person_marker(person_payload: dict) -> None:
    person_id = str(person_payload.get("person_id", "unknown"))
    lat = person_payload.get("lat")
    lon = person_payload.get("lon")
    if lat is None or lon is None:
        return

    color_name = _safe_color(person_payload.get("color", "white"))
    marker_name = f"person-{person_id}"

    previous_marker_id = marker_ids.get(person_id)
    if previous_marker_id:
        try:
            map_view.remove_marker(previous_marker_id)
        except Exception:
            pass

    popup = (
        f"person_id={person_id}<br>"
        f"state={person_payload.get('state', 'unknown')}<br>"
        f"color={color_name}<br>"
        f"entry={person_payload.get('target_entry_id', 'unknown')}"
    )

    marker_id = map_view.add_marker(
        float(lon),
        float(lat),
        name=marker_name,
        color=COLOR_HEX[color_name],
        popup=popup,
    )
    marker_ids[person_id] = marker_id

def _print_dashboard_status() -> None:
    print(
        "dashboard status -> "
        f"people={len(latest_people)} inside_count={inside_count} "
        f"camera_allowed={decision_counts['allowed']} "
        f"camera_denied={decision_counts['denied']}"
    )

def on_message(client, userdata, msg):
    global inside_count, last_render_ts

    topic = msg.topic
    message_counts[topic] += 1

    try:
        payload = json.loads(msg.payload.decode())
    except Exception:
        return

    if topic == person_state_topic:
        person_id = str(payload.get("person_id", "unknown"))
        latest_people[person_id] = payload
        _render_person_marker(payload)
    elif topic == camera_decision_topic:
        allowed = bool(payload.get("allowed", False))
        decision_counts["allowed" if allowed else "denied"] += 1
    elif topic == entry_event_topic:
        event_type = str(payload.get("event_type", "")).lower()
        if event_type == "entered":
            inside_count += 1
        elif event_type == "exited":
            inside_count = max(0, inside_count - 1)

    now = time.time()
    if now - last_render_ts >= 1.0:
        _print_dashboard_status()
        last_render_ts = now

client.on_message = on_message
client.subscribe(person_state_topic, qos=0)
client.subscribe(camera_decision_topic, qos=0)
client.subscribe(entry_event_topic, qos=0)

print("Dashboard subscriptions active.")

Dashboard subscriptions active.


In [ ]:
# Cell purpose: keep dashboard listener active and print periodic lightweight heartbeat
print("Dashboard running. Press Interrupt to stop.")
while True:
    time.sleep(5)
    print(
        "heartbeat -> "
        f"person_msgs={message_counts[person_state_topic]} "
        f"decision_msgs={message_counts[camera_decision_topic]} "
        f"entry_msgs={message_counts[entry_event_topic]} "
        f"inside_count={inside_count}"
    )

In [ ]:
print({"people_seen": len(latest_people), "inside_count": inside_count, "camera_allowed": decision_counts["allowed"], "camera_denied": decision_counts["denied"], "person_msgs": message_counts[person_state_topic], "decision_msgs": message_counts[camera_decision_topic], "entry_msgs": message_counts[entry_event_topic]})